In [0]:
%skip
%pip install -qqq openai mlflow databricks-openai
%restart_python

In [0]:
%run ./common/_utils_config

In [0]:
import json
import mlflow
from datetime import date
from databricks_openai import DatabricksOpenAI

mlflow.openai.autolog()
client = DatabricksOpenAI()
fq = f"{catalog}.{schema}"

print(client.chat.completions.create(
    model=llm_endpoint, messages=[{"role": "user", "content": "Di hola en una frase."}]
).choices[0].message.content)

In [0]:
import json
from pyspark.sql import functions as F

json_types = {"INT": "integer", "BIGINT": "integer", "DOUBLE": "number"}


def load_tools_from_uc():
    loaded_tools = []
    loaded_sql = {}

    routines = spark.sql(f"""
        SELECT routine_name, comment
        FROM {catalog}.information_schema.routines
        WHERE routine_schema = '{schema}'
        ORDER BY routine_name
    """).collect()

    params = spark.sql(f"""
        SELECT specific_name, parameter_name, data_type, comment, ordinal_position
        FROM {catalog}.information_schema.parameters
        WHERE specific_schema = '{schema}' AND parameter_mode = 'IN'
        ORDER BY specific_name, ordinal_position
    """).collect()

    for r in routines:
        fn_params = [p for p in params if p.specific_name == r.routine_name]
        loaded_tools.append({"type": "function", "function": {
            "name": r.routine_name,
            "description": r.comment or r.routine_name,
            "parameters": {
                "type": "object",
                "properties": {
                    p.parameter_name: {
                        "type": json_types.get(p.data_type.upper(), "string"),
                        "description": p.comment or "",
                    }
                    for p in fn_params
                },
                "required": [p.parameter_name for p in fn_params],
            },
        }})
        loaded_sql[r.routine_name] = f"SELECT * FROM {fq}.{r.routine_name}(" + ", ".join(
            f"CAST(:{p.parameter_name} AS {p.data_type.upper()})" for p in fn_params
        ) + ")"

    return loaded_tools, loaded_sql


tools, tool_sql = load_tools_from_uc()


@mlflow.trace(span_type="TOOL")
def execute_tool(tool_name, arguments):
    if tool_name not in tool_sql:
        return json.dumps({"error": f"Unknown tool {tool_name}"})
    try:
        df = spark.sql(tool_sql[tool_name], args=arguments)
        rows = [row[0] for row in df.select(
            F.to_json(F.struct("*"), {"ignoreNullFields": "false"})
        ).collect()]
        return "[" + ",".join(rows) + "]" if rows else "[] (no data for those dates)"
    except Exception as e:
        return json.dumps({"error": str(e)})


for t in tools:
    print(f"- {t['function']['name']}: {t['function']['description'][:90]}")
print(execute_tool("data_coverage", {}))

In [0]:
from datetime import date

# ============================================================
# 1. Agent instructions
# ============================================================
system_prompt = f"""You are an energy assistant for households in Spain, using official Red Eléctrica (REE) data.
Today is {date.today()} ({date.today().strftime('%A')}).
Always use the tools to get figures; never invent prices.
Household prices are PVPC in EUR/kWh. When recommending hours for appliances, look for consecutive cheap hours.
If data for a date is not available yet, say so (tomorrow's prices are published in the evening).

Tool strategy:
- First check whether one existing tool answers the question directly.
- If answering would require calling a tool many times (for example once per day) or combining many results by hand, use create_tool to build ONE tool that answers it in a single query, then call it.
- If NO existing tool can answer the question, use create_tool.
- If create_tool returns an error, fix the SQL and try again (max 2 attempts).
- Never tell the user to look up the data elsewhere; you have the data.

Answer in the user's language, short and practical."""

use_planner = True   # set to False to disable the planner and compare


# ============================================================
# 2. Extract text from the model response
#    (works with any model, including reasoning models that return blocks)
# ============================================================
def extract_text(content):
    if content is None:
        return ""
    if isinstance(content, str):
        return content
    parts = []
    for block in content:
        block_type = block.get("type") if isinstance(block, dict) else getattr(block, "type", None)
        text = block.get("text") if isinstance(block, dict) else getattr(block, "text", None)
        if block_type == "text" and text:
            parts.append(text)
    return "\n".join(parts)


# ============================================================
# 3. Planner: are the existing tools enough?
# ============================================================
def plan_needs_new_tool(question):
    tool_list = "\n".join(
        f"- {t['function']['name']}: {t['function']['description'][:200]}"
        for t in tools if t["function"]["name"] != "create_tool"
    )
    response = client.chat.completions.create(
        model=llm_endpoint,
        messages=[
            {"role": "system", "content": "You decide whether existing tools are enough. Reply only YES or NO."},
            {"role": "user", "content": (
                f"Existing tools:\n{tool_list}\n\n"
                f"Question: {question}\n\n"
                "Can the question be answered fully with at most 2 calls to these existing tools, "
                "without calling a tool once per day? Reply YES or NO."
            )},
        ],
    )
    decision = extract_text(response.choices[0].message.content).strip().upper()
    needs_new = "NO" in decision[:10] and not decision.startswith("YES")
    print(f"[planner] Are existing tools enough? {decision[:20]} → "
          f"{'create a new tool' if needs_new else 'use existing tools'}")
    return needs_new


# ============================================================
# 4. The agent loop
# ============================================================
@mlflow.trace(span_type="AGENT")
def run_agent(question, max_steps=8):
    messages = [{"role": "system", "content": system_prompt},
                {"role": "user", "content": question}]
    tools_used = []
    previous_calls = set()
    force_answer = False
    must_create = plan_needs_new_tool(question) if use_planner else False

    for step in range(max_steps):
        # What the model is allowed to do in this step
        if force_answer:
            choice = "none"                                                     # must answer
        elif must_create and step == 0:
            choice = {"type": "function", "function": {"name": "create_tool"}}  # must create a tool
        else:
            choice = "auto"                                                     # model decides

        try:
            response = client.chat.completions.create(
                model=llm_endpoint, messages=messages, tools=tools, tool_choice=choice,
            )
        except Exception as e:
            if isinstance(choice, dict):
                # Some endpoints don't allow forcing a specific tool: fall back to auto
                print(f"[warning] could not force create_tool ({str(e)[:120]}); continuing in auto mode")
                response = client.chat.completions.create(
                    model=llm_endpoint, messages=messages, tools=tools, tool_choice="auto",
                )
            else:
                raise

        message = response.choices[0].message

        # No tool requests: this is the final answer
        if not message.tool_calls:
            return extract_text(message.content), tools_used

        # Keep in the conversation what the model asked for
        messages.append({
            "role": "assistant",
            "content": extract_text(message.content),
            "tool_calls": [{"id": tc.id, "type": "function",
                            "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                           for tc in message.tool_calls],
        })

        # Run each requested tool
        for tc in message.tool_calls:
            arguments = json.loads(tc.function.arguments or "{}")
            call_key = (tc.function.name, json.dumps(arguments, sort_keys=True))

            if call_key in previous_calls:
                print(f"[step {step + 1}] ⚠️ repeated: {tc.function.name} {arguments} → forcing final answer")
                result = json.dumps({"warning": "You already called this tool with the same arguments. "
                                                "The result is earlier in the conversation. Write the final answer now."})
                force_answer = True
            else:
                previous_calls.add(call_key)
                tools_used.append(tc.function.name)
                print(f"[step {step + 1}] {tc.function.name} {arguments}")
                result = execute_tool(tc.function.name, arguments)

            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

    return "Reached the maximum number of steps without a final answer.", tools_used


print(f"Agent ready · model: {llm_endpoint} · planner: {'ON' if use_planner else 'OFF'} · "
      f"{len(tools)} tools")

In [0]:
%skip
print(run_agent("¿A qué hora salía más barato poner la lavadora el dia 1 de Octubre del 2026? Tarda 2 horas."))

In [0]:
%skip
print(run_agent("¿Cuánto ha costado de media la luz este mes comparado con el mes pasado?"))

In [0]:
%skip
print(run_agent("¿Qué porcentaje de la electricidad fue renovable la semana pasada y qué tecnología produjo más?"))

In [0]:
%skip
print(run_agent("En general, ¿cuáles son las peores horas para cargar el coche eléctrico?"))

**Agent creator**

In [0]:
gold_tables = ["gold_hourly_prices", "gold_daily_generation"]


def describe_table(table_name):
    df = spark.table(f"{fq}.{table_name}")
    parts = []
    for f in df.schema:
        text = f"{f.name} {f.dataType.simpleString()}"
        if f.dataType.simpleString() == "string":
            samples = [r[0] for r in df.select(f.name).distinct().limit(8).collect()]
            text += f" (values e.g. {samples})"
        parts.append(text)
    return f"- {fq}.{table_name}: " + ", ".join(parts)


table_schemas = "\n".join(describe_table(t) for t in gold_tables)
print(table_schemas)

In [0]:
import re

allowed_types = {"DATE", "INT", "BIGINT", "DOUBLE", "STRING"}
forbidden_sql = re.compile(
    r"\b(DROP|DELETE|INSERT|UPDATE|MERGE|ALTER|GRANT|REVOKE|TRUNCATE|CREATE|COPY|VACUUM|OPTIMIZE)\b",
    re.IGNORECASE,
)


def clean_text(text):
    return str(text).replace("'", "").replace("\n", " ")


def validate_tool_spec(function_name, parameters, output_columns, select_sql, test_arguments):
    errors = []

    # Nombre de la herramienta
    if not re.fullmatch(r"[a-z][a-z0-9_]{2,40}", function_name):
        errors.append("function_name must be snake_case, 3-40 chars")
    if function_name in tool_sql:
        errors.append("a tool with that name already exists, use it")

    # Parámetros
    for p in parameters:
        if not re.fullmatch(r"p_[a-z0-9_]{1,30}", p.get("name", "")):
            errors.append(f"parameter {p.get('name')} must start with p_")
        if p.get("type", "").upper() not in allowed_types:
            errors.append(f"parameter type {p.get('type')} not allowed, use {sorted(allowed_types)}")

    # Columnas de salida
    for c in output_columns:
        if not re.fullmatch(r"[a-z][a-z0-9_]{0,40}", c.get("name", "")):
            errors.append(f"output column {c.get('name')} must be snake_case")
        if c.get("type", "").upper() not in allowed_types:
            errors.append(f"output type {c.get('type')} not allowed")

    # SQL: solo lectura y solo sobre las tablas gold
    if not re.match(r"\s*(SELECT|WITH)\b", select_sql, re.IGNORECASE):
        errors.append("select_sql must start with SELECT or WITH")
    if forbidden_sql.search(select_sql) or ";" in select_sql:
        errors.append("select_sql can only read data (no DDL/DML, no ';')")
    if not any(t in select_sql for t in gold_tables):
        errors.append(f"select_sql must read from {gold_tables}")

    # Cada parámetro tiene que usarse dentro del SQL (nada de valores fijos)
    unused = [p.get("name") for p in parameters if not re.search(rf"\b{p.get('name')}\b", select_sql)]
    if unused:
        errors.append(f"parameters {unused} are declared but not used in select_sql; use them instead of hardcoded values")

    # Tiene que haber valores de prueba para cada parámetro
    missing = [p.get("name") for p in parameters if p.get("name") not in (test_arguments or {})]
    if missing:
        errors.append(f"test_arguments must include a value for: {missing}")

    return errors

In [0]:
import json
from datetime import date, timedelta
from pyspark.sql import functions as F


def default_test_arguments(parameters):
    values = {}
    date_count = 0
    for p in parameters:
        param_type = p.get("type", "").upper()
        if param_type == "DATE":
            values[p["name"]] = str(date.today() - timedelta(days=30)) if date_count == 0 else str(date.today())
            date_count += 1
        elif param_type in ("INT", "BIGINT"):
            values[p["name"]] = 30
        elif param_type == "DOUBLE":
            values[p["name"]] = 1.0
        else:
            values[p["name"]] = ""
    return values


def create_tool(function_name, description, parameters, output_columns, select_sql, test_arguments=None, **ignored):
    # Test values: the model's if provided; otherwise generated automatically
    test_arguments = {**default_test_arguments(parameters), **(test_arguments or {})}

    # 1. Automatic validation
    errors = validate_tool_spec(function_name, parameters, output_columns, select_sql, test_arguments)
    if errors:
        print(f"🛡️ Spec rejected by automatic checks: {errors}")
        return json.dumps({
            "status": "rejected",
            "errors": errors,
            "instruction": "Do not resend the same spec. Change it according to the errors, or tell the user you cannot create the tool.",
        })

    # 2. Build the function definition
    params_ddl = ", ".join(
        f"{p['name']} {p['type'].upper()} COMMENT '{clean_text(p.get('description', ''))}'" for p in parameters
    )
    columns_ddl = ", ".join(f"{c['name']} {c['type'].upper()}" for c in output_columns)
    ddl = (
        f"CREATE OR REPLACE FUNCTION {fq}.{function_name}({params_ddl})\n"
        f"RETURNS TABLE ({columns_ddl})\n"
        f"COMMENT '{clean_text(description)}'\n"
        f"RETURN {select_sql}"
    )
    call_sql = f"SELECT * FROM {fq}.{function_name}(" + ", ".join(
        f"CAST(:{p['name']} AS {p['type'].upper()})" for p in parameters
    ) + ")"

    # 3. Create and test BEFORE asking the human; if it fails, remove it and tell the model
    try:
        spark.sql(ddl)
        df_test = spark.sql(call_sql, args=test_arguments).limit(5)
        test_rows = [row[0] for row in df_test.select(
            F.to_json(F.struct("*"), {"ignoreNullFields": "false"})
        ).collect()]
        if not test_rows:
            raise ValueError("Test returned no rows: check filters against the real data.")
        if '":null' in "".join(test_rows):
            raise ValueError(
                "Test returned NULL values: check filter values against the real data "
                "(see 'values e.g.' in the table description)."
            )
    except Exception as e:
        spark.sql(f"DROP FUNCTION IF EXISTS {fq}.{function_name}")
        print(f"🧪 Test failed, the agent will fix it: {str(e)[:150]}")
        return json.dumps({"status": "error", "error": str(e)[:1500], "hint": "Fix the SQL and call create_tool again."})

    # 4. Human approval, only for tools that already work
    print("\n===== THE AGENT PROPOSES A NEW TOOL =====\n")
    print(ddl)
    print("\n🧪 Test passed with:", test_arguments)
    print("Result:", "[" + ",".join(test_rows) + "]")
    if input("\nApprove this tool? (y/n): ").strip().lower() != "y":
        spark.sql(f"DROP FUNCTION IF EXISTS {fq}.{function_name}")
        return json.dumps({"status": "rejected_by_human", "message": "The user did not approve the new tool. Tell the user."})

    # 5. Add it to the agent's toolbox
    tools.append({"type": "function", "function": {
        "name": function_name,
        "description": description,
        "parameters": {
            "type": "object",
            "properties": {
                p["name"]: {
                    "type": "integer" if p["type"].upper() in ("INT", "BIGINT")
                    else "number" if p["type"].upper() == "DOUBLE"
                    else "string",
                    "description": p.get("description", ""),
                }
                for p in parameters
            },
            "required": [p["name"] for p in parameters],
        },
    }})
    tool_sql[function_name] = call_sql

    print(f"\n✅ Tool {function_name} approved and added to the toolbox")
    return json.dumps({
        "status": "created",
        "tool": function_name,
        "test_result": "[" + ",".join(test_rows) + "]",
        "next": f"Now call {function_name} to answer the user.",
    })

In [0]:
# Remove previous versions of create_tool (in case this cell runs more than once)
tools = [t for t in tools if t["function"]["name"] != "create_tool"]

tools.append({"type": "function", "function": {
    "name": "create_tool",
    "description": (
        "Creates a NEW tool (a SQL table function in Unity Catalog) when none of the existing tools can answer. "
        "Only use it if no other tool fits. Available tables:\n" + table_schemas + "\n"
        "SQL dialect is Databricks SQL: DATE_SUB(date, number_of_days) takes an integer (no INTERVAL); "
        "prefer window functions (ROW_NUMBER, RANK) over subqueries; do not put ORDER BY after UNION ALL; keep the SQL simple. "
        "Rules: select_sql is a single SELECT (or WITH ... SELECT) reading those tables; "
        "inside select_sql refer to parameters by name (e.g. p_start), not with ':' and never with hardcoded values; "
        "when filtering text columns use EXACTLY the values shown in 'values e.g.' (same language and spelling); "
        "parameter names start with p_; types: DATE, INT, BIGINT, DOUBLE, STRING; "
        "output_columns must match the SELECT columns in order; write names, descriptions and comments in English. "
        "test_arguments is optional; if omitted, sensible test values are generated automatically."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "function_name": {"type": "string", "description": "snake_case, e.g. weekday_vs_weekend_prices"},
            "description": {"type": "string", "description": "What the tool returns, for future use"},
            "parameters": {"type": "array", "items": {"type": "object", "properties": {
                "name": {"type": "string"},
                "type": {"type": "string"},
                "description": {"type": "string"}},
                "required": ["name", "type"]}},
            "output_columns": {"type": "array", "items": {"type": "object", "properties": {
                "name": {"type": "string"},
                "type": {"type": "string"}},
                "required": ["name", "type"]}},
            "select_sql": {"type": "string"},
            "test_arguments": {"type": "object", "description": "Optional. Values for each parameter to test the tool."},
        },
        "required": ["function_name", "description", "parameters", "output_columns", "select_sql"],
    },
}})

# Tools that are Python functions
python_tools = {"create_tool": create_tool}


@mlflow.trace(span_type="TOOL")
def execute_tool(tool_name, arguments):
    try:
        if tool_name in python_tools:
            return python_tools[tool_name](**arguments)
        if tool_name not in tool_sql:
            return json.dumps({"error": f"Unknown tool {tool_name}"})
        df = spark.sql(tool_sql[tool_name], args=arguments)
        rows = [row[0] for row in df.select(
            F.to_json(F.struct("*"), {"ignoreNullFields": "false"})
        ).collect()]
        return "[" + ",".join(rows) + "]" if rows else "[] (no data for those dates)"
    except Exception as e:
        return json.dumps({"error": str(e)})


print([t["function"]["name"] for t in tools])

In [0]:
result = run_agent("¿Es más barata la luz los fines de semana que entre semana? Compara los últimos 60 días.")
print(result)

In [0]:
from IPython.display import Markdown, display

answer, tools_used = run_agent(
    "What was the most expensive day and the cheapest day for electricity in the last 30 days? "
    "Give me the average price of each."
)
display(Markdown(answer))
print("Tools used:", tools_used)